<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# All Five Sensors

The downward $+z$ __time-of-flight (ToF)__ sensor runs by default. The Duckiedrone carries four more, along $-z$ (up), $+x$ (forward), $-y$ (left), and $+y$ (right), and they stay off until started.

This notebook starts them, checks that their topics appear at the configured rates, and points the subscriber node at all five. That takes one new function and one changed line in the launch file.

Starting the four additional services changes the Duckiedrone's running configuration. Use only a physical Duckiedrone or virtual Duckiematrix entity that you are explicitly authorized to operate.

By the end of the notebook you should be able to answer:

- What has to be running on the Duckiedrone before a topic exists to subscribe to?

- Where do the five topic names come from, and why can they be generated rather than listed?

- Why do the five readings in one printed line come from different measurement times?

## The five sensors

[Table 1](#table-1) combines the five sensors' directions with the configured rates and startup states from [Table 3 of Notebook 2](./2-duckiedrone-tof-architecture.ipynb#table-3).

<a id="table-1"></a>

Table 1: Directions, configured rates, and startup state of the five ToF sensors.

| Sensor | Points | Configured rate ($\mathrm{Hz}$) | Running at boot |
| --- | --- | --- | --- |
| `bottom` | $+z$ (down) | $30$ | Yes |
| `front` | $+x$ (forward) | $5$ | No |
| `left` | $-y$ (left) | $5$ | No |
| `right` | $+y$ (right) | $5$ | No |
| `top` | $-z$ (up) | $5$ | No |

The directions use the Duckiedrone body axes: $+x$ is forward, $+y$ right, and $+z$ down. The camera faces $+x$.

Each sensor runs two containers, a [driver](./2-duckiedrone-tof-architecture.ipynb#the-driver) and a [ROS 2 bridge](./2-duckiedrone-tof-architecture.ipynb#the-ros-2-bridge), so starting the other four adds eight containers. The processor time those cost the Raspberry Pi is why the four stay off by default.

## Activity 1: start the other four sensors

<a id="activity-1"></a>

Start the `duckiedrone/extra_sensors` stack from the base station:

```bash
dts stack up -H DUCKIEDRONE_NAME -d duckiedrone/extra_sensors
```

On the Duckiedrone shell opened as in [Notebook 3](./3-subscribing-to-tof.ipynb#open-a-shell-on-the-duckiedrone), verify that each of the five sensors now has a driver and a bridge:

```bash
docker ps --format '{{.Names}}' | grep tof
```

The output should contain `driver-tof-<sensor>` and `ros2-tof-<sensor>` for every sensor.

Enter a bridge as in [the bridge shell section of Notebook 3](./3-subscribing-to-tof.ipynb#open-a-shell-inside-the-bridge), then list the topics:

```bash
ros2 topic list | grep tof
```

On the example physical Duckiedrone `pdrone24`, the topic list includes:

```shell
/pdrone24/bottom_tof_driver_node/range
/pdrone24/front_tof_driver_node/range
/pdrone24/left_tof_driver_node/range
/pdrone24/right_tof_driver_node/range
/pdrone24/top_tof_driver_node/range
```

[Activity 2](#activity-2) builds these five topic names in code rather than typing them.

Compare the sensor that always runs with one that was just started:

```bash
ros2 topic hz /DUCKIEDRONE_NAME/bottom_tof_driver_node/range
```

```shell
average rate: 30.008
	min: 0.018s max: 0.050s std dev: 0.00404s window: 245
```

```bash
ros2 topic hz /DUCKIEDRONE_NAME/front_tof_driver_node/range
```

```shell
average rate: 5.013
	min: 0.177s max: 0.216s std dev: 0.00688s window: 43
```

Both outputs come from a physical Duckiedrone and match the configured $30\,\mathrm{Hz}$ and $5\,\mathrm{Hz}$. A virtual Duckiedrone can differ, so measure it rather than assume. Ctrl-C stops the command.

## Activity 2: subscribe to all five

<a id="activity-2"></a>

The node takes a list of sensor names and builds each topic with `topics_for`, the function left to write.

Open [packages/solution/solution/tof_math.py](../packages/solution/solution/tof_math.py) and fill in `topics_for` at its `TO IMPLEMENT` block. Every bridge names its topic the same way, so the pattern is the one in the `ros2 topic list` output above, without the `/DUCKIEDRONE_NAME` prefix, which the node adds itself. One list comprehension is enough.

`ToFSubscriber` already calls `topics_for` on its sensor names, so once the function is written, the sensor names are all the node needs.

### Tell the launch file which sensors to use

The node receives its sensor list from the launch file in `packages/tof_listener/launch/`, which `dts code workbench` runs when it starts ([Activity 2 of Notebook 3](./3-subscribing-to-tof.ipynb#activity-2)). The list is therefore set there.

Open [packages/tof_listener/launch/tof_listener_node.launch.py](../packages/tof_listener/launch/tof_listener_node.launch.py) and find the argument that names the sensors:

```python
sensor_names_arg = DeclareLaunchArgument(
    "sensor_names",
    default_value="['bottom']",
    description="Names of the ToF sensors to subscribe to.",
)
```

`default_value` is the one line to change. Name all five, in any order:

```python
    default_value="['bottom','front','left','right','top']",
```

Nothing else changes: the launch file never mentions topics, because `topics_for` builds them. Setting the line back to `['bottom']` restores the node from [Notebook 3](./3-subscribing-to-tof.ipynb).

### Deploy it

The same two commands as [Notebook 3](./3-subscribing-to-tof.ipynb#activity-2):

```bash
dts code build -R DUCKIEDRONE_NAME
dts code workbench -R DUCKIEDRONE_NAME
```

The node now logs five subscriptions at startup, then a line every second with five numbers in it:

```shell
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/bottom_tof_driver_node/range  (sensor: bottom)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/front_tof_driver_node/range  (sensor: front)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/left_tof_driver_node/range  (sensor: left)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/right_tof_driver_node/range  (sensor: right)
[tof_listener_node-1] [INFO] [tof_listener_node]: Subscribing to /pdrone24/top_tof_driver_node/range  (sensor: top)
[tof_listener_node-1] [INFO] [tof_listener_node]: ranges=[bottom=0.013m front=0.116m left=1.288m right=0.483m top=1.660m]
[tof_listener_node-1] [INFO] [tof_listener_node]: ranges=[bottom=0.012m front=0.116m left=1.309m right=0.482m top=1.663m]
```

Check the startup lines first: each topic should be one of the five from [Activity 1](#activity-1), with a publisher in `ros2 topic info`. If the extra sensors stack is down, the four extra subscriptions still succeed but receive nothing, and those sensors are missing from the printed line.

Then read the numbers. Those two lines come from a physical Duckiedrone resting on its legs, one second apart. `bottom` is below the sensor's minimum, as in [Activity 3 of Notebook 3](./3-subscribing-to-tof.ipynb#activity-3), and `front`, `left`, and `right` are whatever surrounds the Duckiedrone.

### Five readings, five different times

The printed line does not show the rate difference. Each entry is the newest message from its sensor, and even a $5\,\mathrm{Hz}$ sensor delivers about five messages per printed line, so every entry can change from line to line. The rates are measured with `ros2 topic hz`, as in [Activity 1](#activity-1).

What the line also hides is that its five numbers were measured at different times. When the line is printed, the `bottom` reading is at most about $33\,\mathrm{ms}$ old and a reading from any of the other four up to $200\,\mathrm{ms}$ old, plus the time each message took to arrive. At $1\,\mathrm{m/s}$, a Duckiedrone moves $20\,\mathrm{cm}$ in $200\,\mathrm{ms}$.

### What did not change

The code from [Notebook 3](./3-subscribing-to-tof.ipynb) did not change. `_make_callback` gives each subscription a callback that carries its sensor's name, and `latest_readings` is keyed by that name, so the five sensors write to five separate entries.

## Going further

Each of these is optional, and each one builds on the node from [Activity 2](#activity-2):

- Subscribe to a subset: `default_value="['bottom','front']"` prints two entries. The node works
  with any number of names.

- Print how old each reading is. Every `ToFReading` carries `stamp_sec`, the time the driver
  read the measurement ([the driver section of Notebook 2](./2-duckiedrone-tof-architecture.ipynb#the-driver)). In
  `cb_log`, subtract it from the current time, `self.get_clock().now().nanoseconds * 1e-9`, and
  print the result next to each range. Compare the age of `bottom` with the age of a sideways
  sensor.

- Watch one sideways sensor on its own while something moves in front of it: `ros2 topic echo
  /DUCKIEDRONE_NAME/front_tof_driver_node/range --field range`.

- Change a sideways sensor's rate in `/data/config/node/tof_driver_left/DUCKIEDRONE_NAME.yaml`, which on
  a physical Duckiedrone belongs to root, so edit it with `sudo`. To slow it down, lower
  `frequency`. Raising it above $5\,\mathrm{Hz}$ needs a shorter `timing_budget` as well:
  `frequency: 10` with `timing_budget: 100` is a valid combination.
  Restart `driver-tof-left` and then
  `ros2-tof-left`, in that order, and confirm the new rate with `ros2 topic hz`.

After the activities, stop the extra-sensors stack from the base-station terminal:

```bash
dts stack down -H DUCKIEDRONE_NAME duckiedrone/extra_sensors
```

## Further reading

[Duckiematrix setup](https://docs.duckietown.com/ente/duckietown-manual/50-duckiematrix/getting-started/duckiematrix-first-steps.html) covers virtual-device preparation for the five-sensor activity; the [sensor_msgs/Range message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Range.msg) describes each published reading.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()